# ⚽ Arsenal FC — Sports Analytics ML Intelligence

This notebook builds and tracks the three core ML models powering the Arsenal Intelligence Platform:

| # | Model | Task | Target |
|---|-------|------|--------|
| 1 | **Attendance Prediction** | Regression | Matchday crowd (0 – 60,704) |
| 2 | **Injury Risk Scoring** | Multi-class Classification | Low / Medium / High |
| 3 | **Match Outcome Prediction** | Multi-class Classification | Win / Draw / Loss |

All runs are tracked in MLflow. Final models expose predictions that align with the `/api/ml/*` endpoints in `server/routes/ml.py`.

In [0]:
%pip install -q scikit-learn==1.4.2 mlflow seaborn category_encoders
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score
)
from mlflow.models.signature import infer_signature

np.random.seed(42)
SNS_PALETTE = "rocket_r"
sns.set_theme(style="darkgrid", palette=SNS_PALETTE)

EXPERIMENT_NAME = "/Users/gokul.pillai@databricks.com/arsenal-ml"
mlflow.set_experiment(EXPERIMENT_NAME)

# Stadium capacity
MAX_CAPACITY = 60_704

print("✅ Imports ready — MLflow experiment:", EXPERIMENT_NAME)

In [0]:
# ── 1. Attendance dataset (500 historical Premier League / cup matches) ──────
rng = np.random.default_rng(42)
n_matches = 500

opponent_tier   = rng.integers(1, 4, n_matches)          # 1=top6, 2=mid, 3=lower
is_weekend      = rng.choice([True, False], n_matches, p=[0.68, 0.32])
temp_c          = rng.normal(12, 6, n_matches).clip(-5, 32)
competition     = rng.choice(
    ["Premier League", "Champions League", "FA Cup", "EFL Cup"],
    n_matches, p=[0.60, 0.20, 0.12, 0.08]
)
booking_vel     = rng.uniform(0.5, 2.0, n_matches)
days_advance    = rng.integers(1, 90, n_matches)
is_rival        = rng.choice([True, False], n_matches, p=[0.15, 0.85])

base_att = 57_000
attendance = (
    base_att
    + (opponent_tier == 1) * 2500
    - (opponent_tier == 3) * 2800
    + is_weekend.astype(int) * 800
    - (temp_c < 3).astype(int) * 700
    + is_rival.astype(int) * 1200
    + (competition == "Champions League") * 1800
    - (competition == "EFL Cup") * 4500
    + (booking_vel - 1.0) * 600
    + rng.normal(0, 400, n_matches)
).clip(42_000, MAX_CAPACITY).astype(int)

df_att = pd.DataFrame({
    "opponent_tier": opponent_tier,
    "is_weekend":    is_weekend.astype(int),
    "temp_c":        temp_c.round(1),
    "competition":   competition,
    "booking_velocity": booking_vel.round(2),
    "days_advance":  days_advance,
    "is_rival":      is_rival.astype(int),
    "attendance":    attendance,
})

# ── 2. Injury risk dataset (1 200 training-session records) ──────────────────
n_ir = 1_200
fatigue         = rng.uniform(10, 95, n_ir)
training_load   = rng.uniform(40, 100, n_ir)
matches_28d     = rng.integers(0, 13, n_ir)
minutes_7d      = rng.uniform(0, 270, n_ir)
age             = rng.integers(18, 35, n_ir)
position        = rng.choice(["GK", "DEF", "MID", "FWD"], n_ir)
days_since_rest = rng.integers(0, 15, n_ir)

risk_score = (
    0.30 * fatigue
    + 0.25 * training_load
    + 2.5  * matches_28d
    + 0.10 * minutes_7d
    + 0.80 * (age > 30).astype(int) * 8
    + 1.2  * days_since_rest
    + rng.normal(0, 6, n_ir)
)
injury_risk = pd.cut(
    risk_score,
    bins=[0, 35, 65, 9999],
    labels=["Low", "Medium", "High"]
).astype(str)

df_ir = pd.DataFrame({
    "fatigue":          fatigue.round(1),
    "training_load":    training_load.round(1),
    "matches_last_28d": matches_28d,
    "minutes_last_7d":  minutes_7d.round(0),
    "age":              age,
    "position":         position,
    "days_since_rest":  days_since_rest,
    "injury_risk":      injury_risk,
})

# ── 3. Match outcome dataset (300 historical matches) ────────────────────────
n_mo = 300
home_adv        = rng.choice([True, False], n_mo, p=[0.55, 0.45])
form_last_5     = rng.uniform(0, 15, n_mo)          # points from last 5 games
xg_rolling      = rng.uniform(0.8, 3.2, n_mo)
goals_conc_avg  = rng.uniform(0.3, 2.0, n_mo)
opp_strength    = rng.integers(1, 6, n_mo)           # 1=weak .. 5=elite
rest_days       = rng.integers(1, 15, n_mo)
injuries_count  = rng.integers(0, 5, n_mo)

win_prob = (
    0.10 * form_last_5
    + 0.15 * xg_rolling
    - 0.12 * goals_conc_avg
    - 0.08 * opp_strength
    + 0.04 * rest_days
    - 0.05 * injuries_count
    + home_adv.astype(int) * 0.40
    + rng.normal(0, 0.25, n_mo)
)
outcome = np.where(win_prob > 1.20, "Win",
          np.where(win_prob > 0.75, "Draw", "Loss"))

df_mo = pd.DataFrame({
    "home_advantage":   home_adv.astype(int),
    "form_last_5":      form_last_5.round(1),
    "xg_rolling":       xg_rolling.round(2),
    "goals_conceded_avg": goals_conc_avg.round(2),
    "opponent_strength":  opp_strength,
    "rest_days":         rest_days,
    "injuries_count":    injuries_count,
    "outcome":           outcome,
})

print(f"Attendance dataset : {df_att.shape}  | target range [{df_att.attendance.min():,} – {df_att.attendance.max():,}]")
print(f"Injury risk dataset: {df_ir.shape}  | classes: {df_ir.injury_risk.value_counts().to_dict()}")
print(f"Match outcome dataset: {df_mo.shape} | classes: {pd.Series(df_mo.outcome).value_counts().to_dict()}")

In [0]:
# ── Feature type table ───────────────────────────────────────────────────
feature_types = pd.DataFrame([
    # Attendance model
    ("opponent_tier",    "Attendance", "Numeric ordinal (1-3)"),
    ("is_weekend",       "Attendance", "Binary"),
    ("temp_c",           "Attendance", "Numeric float (°C)"),
    ("competition",      "Attendance", "Categorical/nominal"),
    ("booking_velocity", "Attendance", "Numeric float"),
    ("days_advance",     "Attendance", "Numeric integer"),
    ("is_rival",         "Attendance", "Binary"),
    # Injury risk model
    ("fatigue",           "Injury Risk", "Numeric float (0-100)"),
    ("training_load",     "Injury Risk", "Numeric float (0-100)"),
    ("matches_last_28d",  "Injury Risk", "Numeric integer"),
    ("minutes_last_7d",   "Injury Risk", "Numeric float"),
    ("age",               "Injury Risk", "Numeric integer"),
    ("position",          "Injury Risk", "Categorical/nominal"),
    ("days_since_rest",   "Injury Risk", "Numeric integer"),
    # Match outcome model
    ("home_advantage",      "Match Outcome", "Binary"),
    ("form_last_5",          "Match Outcome", "Numeric float (points)"),
    ("xg_rolling",           "Match Outcome", "Numeric float"),
    ("goals_conceded_avg",   "Match Outcome", "Numeric float"),
    ("opponent_strength",    "Match Outcome", "Numeric ordinal (1-5)"),
    ("rest_days",            "Match Outcome", "Numeric integer"),
    ("injuries_count",       "Match Outcome", "Numeric integer"),
], columns=["Feature", "Model", "Type"])
display(feature_types)

# ── Target distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Target Variable Distributions", fontsize=14, fontweight="bold")

# 1. Attendance histogram
axes[0].hist(df_att["attendance"], bins=30, color="#EF0107", edgecolor="white", alpha=0.85)
axes[0].axvline(df_att["attendance"].mean(), color="white", linestyle="--", label=f"Mean: {df_att.attendance.mean():,.0f}")
axes[0].set_title("Matchday Attendance", fontweight="bold")
axes[0].set_xlabel("Attendance")
axes[0].legend()

# 2. Injury risk bar
ir_counts = df_ir["injury_risk"].value_counts().reindex(["Low","Medium","High"])
axes[1].bar(ir_counts.index, ir_counts.values, color=["#2ecc71","#f39c12","#e74c3c"], edgecolor="white")
for i, v in enumerate(ir_counts.values):
    axes[1].text(i, v + 8, str(v), ha="center", fontweight="bold")
axes[1].set_title("Injury Risk Classes", fontweight="bold")
axes[1].set_xlabel("Risk Level")

# 3. Match outcome bar
mo_counts = pd.Series(df_mo["outcome"]).value_counts().reindex(["Win","Draw","Loss"])
axes[2].bar(mo_counts.index, mo_counts.values, color=["#27ae60","#95a5a6","#c0392b"], edgecolor="white")
for i, v in enumerate(mo_counts.values):
    axes[2].text(i, v + 2, str(v), ha="center", fontweight="bold")
axes[2].set_title("Match Outcome Classes", fontweight="bold")
axes[2].set_xlabel("Outcome")

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Feature Correlations with Target Variables", fontsize=14, fontweight="bold")

# 1. Attendance: numeric correlations
att_num = df_att.select_dtypes(include="number").copy()
att_corr = att_num.corr()[["attendance"]].drop("attendance").sort_values("attendance")
sns.heatmap(
    att_corr, annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, linewidths=0.5, ax=axes[0], cbar=False
)
axes[0].set_title("Attendance — Feature Correlations", fontweight="bold")
axes[0].set_ylabel("")

# 2. Injury risk: fatigue vs training_load coloured by risk
colour_map = {"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"}
for risk, grp in df_ir.groupby("injury_risk"):
    axes[1].scatter(grp["fatigue"], grp["training_load"],
                    c=colour_map[risk], label=risk, alpha=0.45, s=18)
axes[1].set_xlabel("Fatigue")
axes[1].set_ylabel("Training Load")
axes[1].set_title("Injury Risk — Fatigue vs Training Load", fontweight="bold")
axes[1].legend(title="Risk")

# 3. Match outcome: xG rolling vs form, coloured by outcome
oc_map = {"Win": "#27ae60", "Draw": "#95a5a6", "Loss": "#c0392b"}
for oc, grp in df_mo.groupby("outcome"):
    axes[2].scatter(grp["xg_rolling"], grp["form_last_5"],
                    c=oc_map[oc], label=oc, alpha=0.45, s=18)
axes[2].set_xlabel("xG Rolling Avg")
axes[2].set_ylabel("Form (last 5 pts)")
axes[2].set_title("Match Outcome — xG vs Form", fontweight="bold")
axes[2].legend(title="Outcome")

plt.tight_layout()
plt.show()

# Basic describe
print("\n--- Attendance describe ---")
print(df_att.describe().round(2).to_string())

In [0]:
# ── Helper: build a column transformer for each model ─────────────────────
def make_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")


# ── Model 1: Attendance ────────────────────────────────────────────────────────
ATT_NUM = ["opponent_tier", "is_weekend", "temp_c", "booking_velocity", "days_advance", "is_rival"]
ATT_CAT = ["competition"]

X_att = df_att[ATT_NUM + ATT_CAT]
y_att = df_att["attendance"]

X_att_tr, X_att_tmp, y_att_tr, y_att_tmp = train_test_split(X_att, y_att, test_size=0.30, random_state=42)
X_att_val, X_att_te, y_att_val, y_att_te = train_test_split(X_att_tmp, y_att_tmp, test_size=0.50, random_state=42)
print(f"Attendance  — train:{len(X_att_tr)} | val:{len(X_att_val)} | test:{len(X_att_te)}")

# ── Model 2: Injury Risk ───────────────────────────────────────────────────────
IR_NUM = ["fatigue", "training_load", "matches_last_28d", "minutes_last_7d", "age", "days_since_rest"]
IR_CAT = ["position"]

X_ir = df_ir[IR_NUM + IR_CAT]
y_ir = df_ir["injury_risk"]

X_ir_tr, X_ir_tmp, y_ir_tr, y_ir_tmp = train_test_split(X_ir, y_ir, test_size=0.30, random_state=42)
X_ir_val, X_ir_te, y_ir_val, y_ir_te = train_test_split(X_ir_tmp, y_ir_tmp, test_size=0.50, random_state=42)
print(f"Injury Risk — train:{len(X_ir_tr)} | val:{len(X_ir_val)} | test:{len(X_ir_te)}")

# ── Model 3: Match Outcome ───────────────────────────────────────────────────────
MO_NUM = ["home_advantage", "form_last_5", "xg_rolling", "goals_conceded_avg", "opponent_strength", "rest_days", "injuries_count"]
MO_CAT = []

X_mo = df_mo[MO_NUM]
y_mo = df_mo["outcome"]

X_mo_tr, X_mo_tmp, y_mo_tr, y_mo_tmp = train_test_split(X_mo, y_mo, test_size=0.30, random_state=42, stratify=y_mo)
X_mo_val, X_mo_te, y_mo_val, y_mo_te = train_test_split(X_mo_tmp, y_mo_tmp, test_size=0.50, random_state=42, stratify=y_mo_tmp)
print(f"Match Out.  — train:{len(X_mo_tr)} | val:{len(X_mo_val)} | test:{len(X_mo_te)}")

In [0]:
att_preprocessor = make_preprocessor(ATT_NUM, ATT_CAT)

att_pipeline = Pipeline([
    ("prep",  att_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    )),
])

with mlflow.start_run(run_name="attendance_rf_v1") as run_att:
    mlflow.set_tags({"model": "attendance", "type": "regression", "version": "1"})
    mlflow.log_params({
        "n_estimators": 200, "max_depth": 10, "min_samples_leaf": 4,
        "train_size": len(X_att_tr), "features": ATT_NUM + ATT_CAT,
    })

    att_pipeline.fit(X_att_tr, y_att_tr)

    # Validation metrics
    y_val_pred = att_pipeline.predict(X_att_val)
    mae_val = mean_absolute_error(y_att_val, y_val_pred)
    r2_val  = r2_score(y_att_val, y_val_pred)

    # Test metrics
    y_te_pred = att_pipeline.predict(X_att_te)
    mae_te = mean_absolute_error(y_att_te, y_te_pred)
    r2_te  = r2_score(y_att_te, y_te_pred)

    mlflow.log_metrics({"val_MAE": mae_val, "val_R2": r2_val, "test_MAE": mae_te, "test_R2": r2_te})

    # CV
    cv_mae = -cross_val_score(att_pipeline, X_att_tr, y_att_tr,
                               cv=5, scoring="neg_mean_absolute_error").mean()
    mlflow.log_metric("cv5_MAE", cv_mae)

    # Log model
    sig = infer_signature(X_att_tr, att_pipeline.predict(X_att_tr))
    mlflow.sklearn.log_model(att_pipeline, name="attendance_model", signature=sig,
                              input_example=X_att_tr.head(3))

    ATT_RUN_ID = run_att.info.run_id

print(f"Attendance Model — Val MAE: {mae_val:,.0f} | Val R²: {r2_val:.3f} | Test MAE: {mae_te:,.0f} | Test R²: {r2_te:.3f}")

# ── Feature importance plot ─────────────────────────────────────────────────────────────
rf_model    = att_pipeline.named_steps["model"]
prep_fitted = att_pipeline.named_steps["prep"]
ohe_feats   = list(prep_fitted.named_transformers_["cat"].named_steps["encoder"]
                   .get_feature_names_out(ATT_CAT))
all_feats   = ATT_NUM + ohe_feats

fig, ax = plt.subplots(figsize=(8, 5))
importances = pd.Series(rf_model.feature_importances_, index=all_feats).sort_values()
importances.plot.barh(ax=ax, color="#EF0107", edgecolor="white")
ax.set_title("Attendance Model — Feature Importances", fontweight="bold")
ax.set_xlabel("Mean Decrease Impurity")
plt.tight_layout()
plt.show()

# Actual vs predicted scatter
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_att_te, y_te_pred, alpha=0.5, color="#EF0107", s=20)
ax.plot([y_att_te.min(), y_att_te.max()], [y_att_te.min(), y_att_te.max()], "w--", lw=1.5)
ax.set_xlabel("Actual Attendance")
ax.set_ylabel("Predicted Attendance")
ax.set_title(f"Actual vs Predicted (Test) | MAE={mae_te:,.0f}", fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
ir_preprocessor = make_preprocessor(IR_NUM, IR_CAT)
CLASS_ORDER_IR  = ["Low", "Medium", "High"]

ir_pipeline = Pipeline([
    ("prep",  ir_preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=4,
        subsample=0.85,
        random_state=42,
    )),
])

with mlflow.start_run(run_name="injury_risk_gbt_v1") as run_ir:
    mlflow.set_tags({"model": "injury_risk", "type": "classification", "version": "1"})
    mlflow.log_params({
        "n_estimators": 150, "learning_rate": 0.08, "max_depth": 4, "subsample": 0.85,
        "train_size": len(X_ir_tr), "classes": CLASS_ORDER_IR,
    })

    ir_pipeline.fit(X_ir_tr, y_ir_tr)

    # Metrics
    y_ir_val_pred  = ir_pipeline.predict(X_ir_val)
    y_ir_te_pred   = ir_pipeline.predict(X_ir_te)
    y_ir_te_proba  = ir_pipeline.predict_proba(X_ir_te)

    val_acc = (y_ir_val == y_ir_val_pred).mean()
    te_acc  = (y_ir_te  == y_ir_te_pred ).mean()
    roc_auc = roc_auc_score(
        pd.get_dummies(y_ir_te, columns=CLASS_ORDER_IR),
        y_ir_te_proba, multi_class="ovr", average="macro"
    )

    mlflow.log_metrics({"val_accuracy": val_acc, "test_accuracy": te_acc, "test_roc_auc_macro": roc_auc})

    sig_ir = infer_signature(X_ir_tr, ir_pipeline.predict(X_ir_tr))
    mlflow.sklearn.log_model(ir_pipeline, name="injury_risk_model", signature=sig_ir,
                              input_example=X_ir_tr.head(3))
    IR_RUN_ID = run_ir.info.run_id

print(f"Injury Risk Model — Val Acc: {val_acc:.3f} | Test Acc: {te_acc:.3f} | ROC-AUC macro: {roc_auc:.3f}")
print("\nClassification Report (test):")
print(classification_report(y_ir_te, y_ir_te_pred, target_names=CLASS_ORDER_IR))

# ── Confusion matrix ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_ir_te, y_ir_te_pred, labels=CLASS_ORDER_IR)
Disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER_IR)
Disp.plot(ax=axes[0], cmap="Reds", colorbar=False)
axes[0].set_title("Injury Risk — Confusion Matrix", fontweight="bold")

# Feature importances
ir_model    = ir_pipeline.named_steps["model"]
ir_prep     = ir_pipeline.named_steps["prep"]
ir_ohe      = list(ir_prep.named_transformers_["cat"].named_steps["encoder"]
                   .get_feature_names_out(IR_CAT))
ir_feats    = IR_NUM + ir_ohe
irFI = pd.Series(ir_model.feature_importances_, index=ir_feats).sort_values()
irFI.plot.barh(ax=axes[1], color="#e74c3c", edgecolor="white")
axes[1].set_title("Injury Risk — Feature Importances", fontweight="bold")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

In [0]:
mo_preprocessor = make_preprocessor(MO_NUM, MO_CAT)
CLASS_ORDER_MO  = ["Win", "Draw", "Loss"]

mo_pipeline = Pipeline([
    ("prep",  mo_preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.07,
        max_depth=4,
        subsample=0.80,
        random_state=42,
    )),
])

with mlflow.start_run(run_name="match_outcome_gbt_v1") as run_mo:
    mlflow.set_tags({"model": "match_outcome", "type": "classification", "version": "1"})
    mlflow.log_params({
        "n_estimators": 200, "learning_rate": 0.07, "max_depth": 4, "subsample": 0.80,
        "train_size": len(X_mo_tr), "classes": CLASS_ORDER_MO,
    })

    mo_pipeline.fit(X_mo_tr, y_mo_tr)

    y_mo_val_pred  = mo_pipeline.predict(X_mo_val)
    y_mo_te_pred   = mo_pipeline.predict(X_mo_te)
    y_mo_te_proba  = mo_pipeline.predict_proba(X_mo_te)

    val_acc_mo = (y_mo_val == y_mo_val_pred).mean()
    te_acc_mo  = (y_mo_te  == y_mo_te_pred ).mean()
    roc_auc_mo = roc_auc_score(
        pd.get_dummies(y_mo_te, columns=CLASS_ORDER_MO),
        y_mo_te_proba, multi_class="ovr", average="macro"
    )

    mlflow.log_metrics({"val_accuracy": val_acc_mo, "test_accuracy": te_acc_mo,
                        "test_roc_auc_macro": roc_auc_mo})
    sig_mo = infer_signature(X_mo_tr, mo_pipeline.predict(X_mo_tr))
    mlflow.sklearn.log_model(mo_pipeline, name="match_outcome_model", signature=sig_mo,
                              input_example=X_mo_tr.head(3))
    MO_RUN_ID = run_mo.info.run_id

print(f"Match Outcome Model — Val Acc: {val_acc_mo:.3f} | Test Acc: {te_acc_mo:.3f} | ROC-AUC macro: {roc_auc_mo:.3f}")
print("\nClassification Report (test):")
print(classification_report(y_mo_te, y_mo_te_pred, target_names=mo_pipeline.classes_))

# ── Confusion matrix + feature importances ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_mo = confusion_matrix(y_mo_te, y_mo_te_pred, labels=mo_pipeline.classes_)
Disp2 = ConfusionMatrixDisplay(cm_mo, display_labels=mo_pipeline.classes_)
Disp2.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Match Outcome — Confusion Matrix", fontweight="bold")

mo_model = mo_pipeline.named_steps["model"]
moFI = pd.Series(mo_model.feature_importances_, index=MO_NUM).sort_values()
moFI.plot.barh(ax=axes[1], color="#27ae60", edgecolor="white")
axes[1].set_title("Match Outcome — Feature Importances", fontweight="bold")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()